<a href="https://colab.research.google.com/github/c-marq/AI-Thinking-CAI1001C/blob/main/08-Classification-Part-2/Guided-Project/GP07_08_Classification_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GP07-08: Classification — From Sorting to Deciding

## CAI1001C: Artificial Intelligence Thinking
**Chapters 7 & 8 — Guided Demo**

---

### What We're Doing Today

Classification is the most common supervised learning task in industry. Instead of predicting a number (regression, Chapter 6), we're predicting a **category** — yes/no, spam/not spam, disease/no disease.

Today we'll build and compare **four** classification algorithms on a real heart disease dataset:
- **Part 1:** k-Nearest Neighbors (k-NN) — *ask your neighbors*
- **Part 2:** Decision Trees — *follow a flowchart of questions*
- **Part 3:** Logistic Regression & SVMs — *draw a boundary line*
- **Part 4:** The Four-Classifier Showdown — *compare them all*

### Learning Objectives

By the end of this demo, you will be able to:
1. Train a k-NN classifier and experiment with different values of K
2. Train a decision tree and visualize its decision logic
3. Run logistic regression and SVM classifiers
4. Compare all four classifiers using accuracy, precision, and recall
5. Explain why accuracy alone can be misleading

### How This Demo Works

- **Run every cell in order** — each builds on the previous one
- Look for **▶ MODIFY** markers — these are spots where you'll change a value and re-run to see what happens
- Look for **💡 Interpretation** cells — read these to understand what the output means
- The final section has `# YOUR CODE HERE` — that's your turn to apply what you learned

---

## Setup: Load Data and Import Libraries

▸ **Run this cell. Do not modify.**

In [ ]:
# ============================================
# SETUP — Run this cell first. Do not modify.
# ============================================

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score

# Load the heart disease dataset (302 patients, 7 features + 1 label)
url = "https://raw.githubusercontent.com/c-marq/AI-Thinking-CAI1001C/refs/heads/main/07-Classification-Part-1/Datasets/heart_disease_patients.csv"
df = pd.read_csv(url)

print(f"Dataset loaded: {df.shape[0]} patients, {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
df.head()

### Quick Look at the Data

Before we build any models, let's understand what we're working with. Each row is a patient. The features describe their health profile. The label (`heart_disease`) is what we're trying to predict: **1 = has heart disease, 0 = does not**.

▸ **Run this cell. Do not modify.**

In [ ]:
# Quick data exploration — Do not modify
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"\nTotal patients: {len(df)}")
print(f"Heart disease (1): {df['heart_disease'].sum()} ({df['heart_disease'].mean():.1%})")
print(f"No disease (0):    {(df['heart_disease'] == 0).sum()} ({(df['heart_disease'] == 0).mean():.1%})")
print(f"\nMale patients:   {(df['sex'] == 1).sum()} ({(df['sex'] == 1).mean():.1%})")
print(f"Female patients: {(df['sex'] == 0).sum()} ({(df['sex'] == 0).mean():.1%})")

# Heart disease rate by sex
print(f"\nHeart disease rate — Males:   {df[df['sex']==1]['heart_disease'].mean():.1%}")
print(f"Heart disease rate — Females: {df[df['sex']==0]['heart_disease'].mean():.1%}")

print(f"\nFeature ranges:")
for col in df.columns[:-1]:
    print(f"  {col}: {df[col].min()} to {df[col].max()}")

### 💡 Interpretation

Notice two things:
1. **The dataset is 68% male.** The model will see twice as many male patients during training. We'll come back to why that matters.
2. **Feature scales vary wildly.** Cholesterol goes up to 564, while exercise_angina is just 0 or 1. This will matter for algorithms that measure distance.

Now let's prepare the data for modeling.

### Prepare: Features, Labels, and Train/Test Split

▸ **Run this cell. Do not modify.**

In [ ]:
# Separate features (X) and label (y)
X = df.drop(columns='heart_disease')
y = df['heart_disease']  # 1 = heart disease, 0 = no disease

# Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features (needed for k-NN, logistic regression, and SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} patients")
print(f"Test set:     {X_test.shape[0]} patients")
print(f"\nTest set breakdown:")
print(f"  Heart disease: {(y_test == 1).sum()}")
print(f"  No disease:    {(y_test == 0).sum()}")

### 💡 Why Scaling?

We created **two versions** of the training data:
- `X_train` / `X_test` — original values (for decision trees)
- `X_train_scaled` / `X_test_scaled` — standardized values (for k-NN, logistic regression, SVM)

**Why?** Algorithms that measure *distance* between points (k-NN) or assign *weights* to features (logistic regression, SVM) are thrown off when features have wildly different scales. Cholesterol (up to 564) would overpower exercise_angina (0 or 1) in distance calculations. Scaling puts all features on equal footing.

Decision trees don't care about scale — they split on thresholds ("Is cholesterol > 250?"), so the magnitude doesn't matter.

---

# Part 1: k-Nearest Neighbors (k-NN)

### The Idea

To classify a new patient, k-NN finds the **K closest patients** in the training data and takes a **majority vote**. If 3 out of 5 nearest neighbors have heart disease, the prediction is "heart disease."

*"Dime con quién andas y te diré quién eres."* — Tell me who you walk with, and I'll tell you who you are.

▸ **Run this cell. Do not modify.**

In [ ]:
# Train k-NN with K=5 (using SCALED data)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# Predict and evaluate
knn_predictions = knn.predict(X_test_scaled)
knn_accuracy = accuracy_score(y_test, knn_predictions)

print(f"k-NN Accuracy (K=5): {knn_accuracy:.2%}")

### 💡 Interpretation

**80.33% accuracy** — the model correctly classified about 4 out of 5 patients. Not bad for a first try with default settings!

But is K=5 the best choice? Let's find out.

---

### ▶ MODIFY: Test Different K Values

The next cell tests K = 1, 3, 5, 7, 9, 11 and plots accuracy for each.

▸ **Run this cell first as-is**, then modify it:
- **Change 1:** Add K=15 and K=21 to the `k_values` list
- **Change 2:** Re-run and observe — does the downward trend continue?

In [ ]:
# ▶ MODIFY: Add more K values to this list and re-run
k_values = [1, 3, 5, 7, 9, 11]

k_accuracies = []

for k in k_values:
    knn_temp = KNeighborsClassifier(n_neighbors=k)
    knn_temp.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, knn_temp.predict(X_test_scaled))
    k_accuracies.append(acc)
    print(f"k-NN Accuracy (K={k:>2}): {acc:.2%}")

# Find the best K
best_k = k_values[k_accuracies.index(max(k_accuracies))]
print(f"\n🏆 Best K: {best_k} with accuracy {max(k_accuracies):.2%}")

# Plot K vs Accuracy
plt.figure(figsize=(8, 4))
plt.plot(k_values, k_accuracies, marker='o', color='coral', linewidth=2, markersize=8)
plt.xlabel('K (Number of Neighbors)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('k-NN: How K Affects Accuracy', fontsize=14)
plt.xticks(k_values)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 💡 Interpretation

- **K=1** is the worst (73.77%) — asking only one neighbor makes the model overreact to noise
- **K=3** is the best (81.97%) — a small, focused group gives the clearest signal
- As K increases past the sweet spot, accuracy drifts down — too many neighbors blurs the local pattern

**Rule of thumb:** Always use an odd K with two classes (avoids ties). The sweet spot is usually somewhere between 3 and 11.

---

# Part 2: Decision Trees

### The Idea

A decision tree classifies by asking a series of **yes/no questions** about features, one at a time. Think of an ER triage nurse: *Chest pain? → Yes → Fast-track to cardiology. No → Fever above 102? → ...*

The algorithm learns **which questions to ask first** from the training data — the feature that best separates heart disease from no heart disease goes at the top.

**Key difference from k-NN:** Decision trees use the **unscaled** data. They split on thresholds ("Is cholesterol > 250?"), so scale doesn't matter.

▸ **Run this cell. Do not modify.**

In [ ]:
# Train a decision tree (using UNSCALED data — trees don't need scaling)
tree = DecisionTreeClassifier(max_depth=4, random_state=42)
tree.fit(X_train, y_train)

# Predict and evaluate
tree_predictions = tree.predict(X_test)
tree_accuracy = accuracy_score(y_test, tree_predictions)

print(f"Decision Tree Accuracy (max_depth=4): {tree_accuracy:.2%}")

### 💡 Interpretation

**80.33%** — the same as k-NN with K=5. But these two models are making different predictions for different patients. Same accuracy doesn't mean same reasoning.

---

### ▶ MODIFY: Change Tree Depth

The `max_depth` parameter controls how many questions deep the tree can go. Deeper = more complex.

▸ **Change `max_depth` to different values and re-run:**
- Try `max_depth=2` (very simple tree)
- Try `max_depth=6` (more complex)
- Try `max_depth=None` (no limit — the tree grows as deep as it wants)

**Question to think about:** Does deeper always mean better? What might go wrong with `max_depth=None`?

In [ ]:
# ▶ MODIFY: Change max_depth and re-run
TREE_DEPTH = 4  # ← Change this value: try 2, 3, 5, 6, None

tree_mod = DecisionTreeClassifier(max_depth=TREE_DEPTH, random_state=42)
tree_mod.fit(X_train, y_train)

mod_accuracy = accuracy_score(y_test, tree_mod.predict(X_test))
print(f"Decision Tree Accuracy (max_depth={TREE_DEPTH}): {mod_accuracy:.2%}")

### Visualize: See How the Tree Thinks

This is why hospitals love decision trees — you can read the model's reasoning.

▸ **Run this cell. Do not modify.**

In [ ]:
# Visualize the decision tree
plt.figure(figsize=(16, 8))
plot_tree(
    tree,
    feature_names=X.columns.tolist(),
    class_names=['No Disease', 'Heart Disease'],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title('Decision Tree: Heart Disease Diagnosis Logic', fontsize=14)
plt.tight_layout()
plt.show()

# Show feature importances
print("\n🔍 Feature Importances (which features matter most):")
print("-" * 45)
for name, imp in sorted(zip(X.columns, tree.feature_importances_),
                         key=lambda x: x[1], reverse=True):
    bar = "█" * int(imp * 40)
    print(f"  {name:<20} {imp:.3f}  {bar}")

### 💡 Interpretation

The tree reveals its reasoning:
- **chest_pain_type** is the most important feature (0.388) — the first question the tree asks
- **age** (0.134) and **sex** (0.131) follow
- You can trace any patient through the tree and see *exactly* why they got their prediction

This is the tradeoff: **k-NN is a black box** ("your neighbors voted this way"), but **decision trees show their work**. When would each matter more?

---

### k-NN vs. Decision Tree: Side by Side

▸ **Run this cell. Do not modify.**

In [ ]:
# Head-to-head comparison: k-NN (K=3) vs Decision Tree (depth=4)
knn_best = KNeighborsClassifier(n_neighbors=3)
knn_best.fit(X_train_scaled, y_train)
knn_preds = knn_best.predict(X_test_scaled)

tree_preds = tree.predict(X_test)

print("=" * 50)
print("k-NN Classification Report (K=3)")
print("=" * 50)
print(classification_report(y_test, knn_preds,
      target_names=['No Disease', 'Heart Disease']))

print("=" * 50)
print("Decision Tree Classification Report (depth=4)")
print("=" * 50)
print(classification_report(y_test, tree_preds,
      target_names=['No Disease', 'Heart Disease']))

# Quick comparison
print("\n📊 Quick Comparison:")
print(f"  k-NN (K=3):         {accuracy_score(y_test, knn_preds):.2%} accuracy")
print(f"  Decision Tree (d=4): {accuracy_score(y_test, tree_preds):.2%} accuracy")
print(f"  Winner: {'k-NN' if accuracy_score(y_test, knn_preds) > accuracy_score(y_test, tree_preds) else 'Decision Tree' if accuracy_score(y_test, tree_preds) > accuracy_score(y_test, knn_preds) else 'Tie!'}")

---

# Part 3: Two More Classifiers — Logistic Regression & SVM

We've now built two classifiers. But the toolbox has more options. Let's add two more:

- **Logistic Regression** — draws a straight decision line. Everything on one side = heart disease, the other side = no heart disease. Simple, fast, and surprisingly effective.
- **Support Vector Machine (SVM)** — also draws a line, but specifically finds the line with the **widest margin** (the most breathing room) between the two classes.

Both use **scaled** data, just like k-NN.

▸ **Run this cell. Do not modify.**

In [ ]:
# Train Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
lr_accuracy = accuracy_score(y_test, lr.predict(X_test_scaled))
print(f"Logistic Regression Accuracy: {lr_accuracy:.2%}")

# Train SVM with linear kernel
svm = SVC(kernel='linear', random_state=42)
svm.fit(X_train_scaled, y_train)
svm_accuracy = accuracy_score(y_test, svm.predict(X_test_scaled))
print(f"SVM (linear kernel) Accuracy: {svm_accuracy:.2%}")

print(f"\nBoth match k-NN at {lr_accuracy:.2%} — three-way tie!")
print("But accuracy alone doesn't tell the whole story...")

### ▶ MODIFY: Try Different SVM Kernels

SVM can draw different types of boundaries. A `linear` kernel draws a straight line. An `rbf` kernel can curve. A `poly` kernel creates polynomial boundaries.

▸ **Change the kernel and re-run:**

In [ ]:
# ▶ MODIFY: Change the kernel — try 'linear', 'rbf', or 'poly'
SVM_KERNEL = 'linear'  # ← Change this value

svm_mod = SVC(kernel=SVM_KERNEL, random_state=42)
svm_mod.fit(X_train_scaled, y_train)
svm_mod_accuracy = accuracy_score(y_test, svm_mod.predict(X_test_scaled))

print(f"SVM ({SVM_KERNEL} kernel) Accuracy: {svm_mod_accuracy:.2%}")
print(f"\n💡 More complex isn't always better — 'linear' often wins on smaller datasets.")

---

# Part 4: The Four-Classifier Showdown

This is the payoff. Same data, same split, same scaling — four different algorithms. We'll compare them on **three metrics**:

- **Accuracy** — overall, how many predictions were correct?
- **Precision** — when the model says "heart disease," how often is it right?
- **Recall** — of all patients who *actually* have heart disease, how many did the model catch?

Think of it like a bouncer at a club:
- **High precision** = rarely lets in someone who shouldn't be there (few false alarms)
- **High recall** = makes sure every VIP gets in (catches all the real cases)

▸ **Run this cell. Do not modify.**

In [ ]:
# ============================================
# THE FOUR-CLASSIFIER SHOWDOWN
# Same data. Same split. Four algorithms.
# ============================================

# Train all four
knn_final = KNeighborsClassifier(n_neighbors=3)
knn_final.fit(X_train_scaled, y_train)
knn_pred = knn_final.predict(X_test_scaled)

dt_final = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_final.fit(X_train, y_train)
dt_pred = dt_final.predict(X_test)

lr_final = LogisticRegression(max_iter=1000, random_state=42)
lr_final.fit(X_train_scaled, y_train)
lr_pred = lr_final.predict(X_test_scaled)

svm_final = SVC(kernel='linear', random_state=42)
svm_final.fit(X_train_scaled, y_train)
svm_pred = svm_final.predict(X_test_scaled)

# Build the comparison table
results = pd.DataFrame({
    'Classifier': ['k-NN (K=3)', 'Decision Tree (depth=4)',
                   'Logistic Regression', 'SVM (linear)'],
    'Accuracy (%)': [
        round(accuracy_score(y_test, p) * 100, 2)
        for p in [knn_pred, dt_pred, lr_pred, svm_pred]
    ],
    'Precision (%)': [
        round(precision_score(y_test, p) * 100, 2)
        for p in [knn_pred, dt_pred, lr_pred, svm_pred]
    ],
    'Recall (%)': [
        round(recall_score(y_test, p) * 100, 2)
        for p in [knn_pred, dt_pred, lr_pred, svm_pred]
    ]
})

print("🏆 THE FOUR-CLASSIFIER COMPARISON TABLE")
print("=" * 65)
print(results.to_string(index=False))
print("=" * 65)

# Count false negatives (missed heart disease patients)
print(f"\n❌ Missed Heart Disease Cases (False Negatives) out of {(y_test==1).sum()}:")
for name, pred in zip(['k-NN', 'Decision Tree', 'Log. Regression', 'SVM'],
                       [knn_pred, dt_pred, lr_pred, svm_pred]):
    fn = ((y_test == 1) & (pred == 0)).sum()
    print(f"  {name:<16} missed {fn} patients")

### 💡 Interpretation — What the Table Reveals

If you only looked at **accuracy**, you'd shrug — three classifiers tied at 81.97%, decision tree slightly behind. Boring.

But **precision and recall tell a completely different story:**

| What It Means | Winner | The Tradeoff |
|---|---|---|
| **Best at catching sick patients** (highest recall) | k-NN (81.25%) — missed only 6 | More false alarms (5 healthy patients flagged) |
| **Most reliable positive predictions** (highest precision) | Log. Regression & SVM (88.89%) | Missed 8 sick patients |
| **Most explainable** | Decision Tree | Lowest overall accuracy |

**The key question:** In a hospital, which is worse — telling a healthy person "we need more tests" (false positive), or telling a sick person "you're fine" (false negative)?

Most doctors would choose the classifier with the **highest recall** — missing a heart disease case is more dangerous than running an extra test.

> **This is the model selection lesson:** There is no best algorithm. There is only the best algorithm *for your problem*. That professional judgment is the skill.

---

# Your Turn: Change the Rules

### ▶ MODIFY: What Happens With a Different Split?

The results above used an 80/20 train/test split. But what if we change that? A different split means different patients in the test set, which can change the rankings.

▸ **Change `NEW_TEST_SIZE` below and re-run.** Try 0.25 and 0.30. Do the rankings shift?

In [ ]:
# ▶ MODIFY: Change the test size and re-run
NEW_TEST_SIZE = 0.2  # ← Try 0.25 or 0.30

# YOUR CODE HERE: Re-split the data with the new test size
# Hint: Copy the train_test_split and StandardScaler code from the Setup,
# but use NEW_TEST_SIZE instead of 0.2

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X, y, test_size=NEW_TEST_SIZE, random_state=42
)

scaler2 = StandardScaler()
X_train2_scaled = scaler2.fit_transform(X_train2)
X_test2_scaled = scaler2.transform(X_test2)

# Re-train and compare all four classifiers
classifiers = {
    'k-NN (K=3)': KNeighborsClassifier(n_neighbors=3),
    'Decision Tree (d=4)': DecisionTreeClassifier(max_depth=4, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'SVM (linear)': SVC(kernel='linear', random_state=42)
}

print(f"Test size: {NEW_TEST_SIZE} ({X_test2.shape[0]} test patients)")
print("=" * 65)

for name, clf in classifiers.items():
    # Decision tree uses unscaled data
    if 'Tree' in name:
        clf.fit(X_train2, y_train2)
        pred = clf.predict(X_test2)
    else:
        clf.fit(X_train2_scaled, y_train2)
        pred = clf.predict(X_test2_scaled)

    acc = accuracy_score(y_test2, pred)
    prec = precision_score(y_test2, pred)
    rec = recall_score(y_test2, pred)
    print(f"  {name:<25} Acc: {acc:.2%}  Prec: {prec:.2%}  Rec: {rec:.2%}")

print("=" * 65)
print("\n💡 Did the rankings change? That's the point — a single accuracy")
print("   number on a single split is never the final word.")

---

## What You Just Learned

1. **Classification** predicts categories (disease/no disease), not numbers
2. **k-NN** classifies by finding similar examples and taking a majority vote — K=3 worked best here
3. **Decision trees** classify by asking yes/no questions about features — they show their reasoning
4. **Logistic regression** draws a probability-based decision line; **SVMs** find the line with the widest margin
5. **Accuracy alone is misleading** — precision and recall reveal *who pays* for the model's errors
6. **No single algorithm is always best** — the right choice depends on the problem and what errors you can afford

### ➡️ Next: Group Lab

You're about to switch from coding to exploring. In the group lab, you'll use a visual tool to interact with classification concepts — no code required. The goal: build intuition for how these algorithms work by *seeing* them in action.

---

*GP07-08 | CAI1001C: AI Thinking | Chapters 7 & 8 | Miami Dade College*